# CA2, Phase 2: the performance anomaly and airtime fairness

This notebook is **optional**: any language or tool is fine for the analysis (see the setup section of the assignment). It only **loads** simulation results; the models, the analysis and the plots are yours to write, in new cells under each section heading.

Rules:
- Your group's parameters come from `../params.json`. Station `host[0]` is the slow one.
- Every number and plot in your report must be reproducible from the code you hand in; the TAs do not run it, but at the in-person delivery you will rerun parts of it and change them.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from omnetpp.scave import results

with open(os.path.join("..", "params.json")) as f:
    P = json.load(f)
L, N = P["L"], P["n2"]
RATES = [P["Rslow"]] + [P["Rfast"]] * (N - 1)   # Mbps, host[0] first
T_MEAS = 10.0                                     # s: 12 s simulated, 2 s warm-up
print(f"n = {N}, L = {L} B, bitrates = {RATES} Mbps")

## Loading results

`load_scalars(config)` returns every scalar of one config, with iteration variables and `repetition` as columns.
`throughput(config, direction)` returns one row per run and one column per station, in Mbps: for `"up"` it is what the server received from each station, for `"down"` what each station received.
`delay(config)` returns the mean and maximum end-to-end delay per station (downlink).

In [ ]:
RESULTS = "results"


def load(config):
    df = results.read_result_files(os.path.join(RESULTS, f"{config}-*.sca"))
    sc = results.get_scalars(df, include_itervars=True, include_runattrs=True)
    st = results.get_histograms(df, include_itervars=True, include_runattrs=True)
    for t in (sc, st):
        t["repetition"] = pd.to_numeric(t["repetition"])
    return sc, st


def throughput(config, direction):
    sc, _ = load(config)
    cols = {}
    for i in range(N):
        module = f"Anomaly.server.app[{i}]" if direction == "up" else f"Anomaly.host[{i}].app[0]"
        rows = sc[(sc.module == module) & (sc.name == "packetReceived:count")]
        cols[f"host[{i}]"] = rows.set_index("repetition").value * 8 * L / T_MEAS / 1e6
    return pd.DataFrame(cols)


def delay(config):
    _, st = load(config)
    d = st[st.name == "endToEndDelay:histogram"]
    return d.groupby("module")[["mean", "max"]].agg({"mean": "mean", "max": "max"})

Example: per-station uplink throughput (run the `Uplink` config first).

In [ ]:
throughput("Uplink", "up")

## 2-1  The anomaly, uplink and downlink FIFO

## 2-2  Your airtime-fair scheduler

## 2-3  Analysis and limits